# **ETL (Extract-Transform-Load)**

## **0. Environment setup**

In [1]:
import os
import gdown
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Set display options
pd.set_option("display.width", 100)
plt.style.use("seaborn-v0_8")

# For reproductibility
np.random.seed(42)

In [3]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

## **1. Data loading**

#### **1.0. Extracting the data**

In [ ]:
# Download data
file_id = "1GffOYmcAMP17oi2BwC7Dp4l5F7rEjHRr" 
url = f"https://drive.google.com/uc?id={file_id}"
output = "mind_large.zip"
is_downloaded = False

for file in os.listdir("."):
    if file.startswith("mind_large"):
        print("mind_large is already downloaded.")
        is_downloaded = True
        break
        
if is_downloaded == False:
    print("Downloading mind_large.zip ...")
    gdown.download(url, output, quiet=False)
    print("Download complete!")

    # Extract compressed file
    with zipfile.ZipFile("mind_large.zip", "r") as z:
        z.extractall(".")

mind_large already downloaded.


#### **1.1. Defining the different sets**

In [5]:
sets = ["train", "dev", "test"]

#### **1.2. Loading news data**

In [6]:
%%time

news_header = ["id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
news = {}

for _set in sets:
    news[_set] = pd.read_csv(f"mind_large/news_{_set}.tsv", names=news_header, sep="\t")

CPU times: user 4.41 s, sys: 543 ms, total: 4.95 s
Wall time: 4.96 s


In [7]:
news["train"].head()

,id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N88753,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N45436,news,newsscienceandtechnology,Walmart Slashes Prices on Last-Generation iPads,Apple's new iPad releases bring big deals on l...,https://assets.msn.com/labs/mind/AABmf2I.html,"[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ...","[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ..."
2,N23144,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
3,N86255,health,medical,Dispose of unwanted prescription drugs during ...,NaN,https://assets.msn.com/labs/mind/AAISxPN.html,"[{""Label"": ""Drug Enforcement Administration"", ...",[]
4,N93187,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."


In [8]:
all_news_df = pd.concat([news["train"], news["dev"], news["test"]], ignore_index=True)    
unique_article_count = all_news_df["id"].nunique()
print(f"Total Unique News Articles : {unique_article_count}")

Total Unique News Articles : 130379


#### **1.3. Loading behaviors data**

In [9]:
%%time

behaviors_header = ["impression_id", "user_id", "time", "history", "impressions"]
behaviors = {}

for _set in sets:
    behaviors[_set] = pd.read_csv(f"mind_large/behaviors_{_set}.tsv", names=behaviors_header, sep="\t")

CPU times: user 32.8 s, sys: 3.56 s, total: 36.4 s
Wall time: 36.5 s


In [10]:
behaviors["train"].head()

,impression_id,user_id,time,history,impressions
0,1,U87243,11/10/2019 11:30:54 AM,N8668 N39081 N65259 N79529 N73408 N43615 N2937...,N78206-0 N26368-0 N7578-0 N58592-0 N19858-0 N5...
1,2,U598644,11/12/2019 1:45:29 PM,N56056 N8726 N70353 N67998 N83823 N111108 N107...,N47996-0 N82719-0 N117066-0 N8491-0 N123784-0 ...
2,3,U532401,11/13/2019 11:23:03 AM,N128643 N87446 N122948 N9375 N82348 N129412 N5...,N103852-0 N53474-0 N127836-0 N47925-1
3,4,U593596,11/12/2019 12:24:09 PM,N31043 N39592 N4104 N8223 N114581 N92747 N1207...,N38902-0 N76434-0 N71593-0 N100073-0 N108736-0...
4,5,U239687,11/14/2019 8:03:01 PM,N65250 N122359 N71723 N53796 N41663 N41484 N11...,N76209-0 N48841-0 N67937-0 N62235-0 N6307-0 N3...


In [11]:
all_behaviors_df = pd.concat([behaviors["train"], behaviors["dev"], behaviors["test"]], ignore_index=True)    
unique_behaviors_count = all_behaviors_df["impression_id"].nunique()
print(f"Total Unique Behaviors (Impressions) : {unique_behaviors_count}")

Total Unique Behaviors (Impressions) : 2370727


## **2. Preprocessing**

In [12]:
# Remove the **url**, **title_entities** and **abstract_entities** columns from news datasets.
# Handle missing values (in case abstract is null and title exists, replace abstract by title instead of dropping it)
# Handle duplicated values

## **3. Adding data in Supabase**

In [13]:
# Add only news data in the database (behaviors are just used offline)